# 00 — Exemplo de leitura

Conecta **somente leitura** (`src/notebooks.py`) e mostra os 10 números de referência a partir das tabelas **públicas** (`gold.fato_mercado_uf`, `gold.execucao` e a versão vigente das premissas). Sem o banco local, usa o conjunto público (modo demonstração).

Regras: notebook lê, não escreve; `interno_*` só em `notebooks/privado/`; saídas limpas antes do commit (ver `notebooks/README.md`).

In [ ]:
import json
import sys
from pathlib import Path

raiz = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(raiz))
from src.notebooks import conectar, ler

con = conectar()

In [ ]:
r = ler("SELECT * FROM gold.fato_mercado_uf WHERE nivel = 'base' AND regra_classe = 'v2' AND cenario_stf = 'mantem'", con=con).iloc[0]
premissas = json.loads(ler("SELECT conteudo_json FROM estado.premissas_versoes ORDER BY versao DESC LIMIT 1", con=con).iloc[0, 0])
taxa = premissas["premissas"]["taxa_seguro_anual"]["valores"]["base"]
carga = ler("SELECT max(executado_em) FROM gold.execucao", con=con).iloc[0, 0]

numeros = {
    "Serventias ativas": r.n_serventias_ativas,
    "Empregados CLT": r.clt,
    "Obrigadas (≈)": r.n_universo,
    "Passivo de todas (R$)": r.passivo_todas,
    "Passivo das obrigadas (R$)": r.passivo_universo,
    "Com déficit (≈)": r.n_deficit,
    "Déficit (R$)": r.valor_descoberto,
    "IS do mercado (R$)": r.mercado_importancia_segurada,
    "Prêmio anual do mercado (R$)": r.mercado_premio_emitido,
    "Teto teórico (R$)": r.passivo_universo * taxa,
}
print(f"Carga de {carga:%d/%m/%Y}, premissas v{premissas['versao']}")
numeros

In [ ]:
con.close()